# SLOWTRACE X SPHEREx survey
Use a private Kaggle CPU notebook. Attach the private source dataset containing `slowtrace-kaggle-source.zip`, turn on Internet, and attach the two Supabase secrets through Add-ons → Secrets. The first saved run pauses after seven committed products so a second saved run can verify resumption. Keep scheduling off until both runs are checked.

In [ ]:
MODE = 'pilot'
PILOT_STOP_AFTER = 7  # Set to None for the second pilot run.
PILOT_VERIFIED = False
BATCH_PRODUCTS = 20
BATCHES = 1
MINUTES = 45
RAM_MIB = 8192
SCRATCH_MIB = 2048
TRANSFER_MIB = 2048

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from zipfile import ZipFile

import requests
from kaggle_secrets import UserSecretsClient

assert Path('/kaggle/working').is_dir() and os.environ.get('KAGGLE_KERNEL_RUN_TYPE'), 'Run this in Kaggle.'
assert not os.environ.get('CUDA_VISIBLE_DEVICES', '').strip('-1'), 'Set the accelerator to None.'
assert MODE == 'pilot' or PILOT_VERIFIED, 'Verify the two pilot runs before enabling survey mode.'
archive = 'https://nasa-irsa-spherex.s3.us-east-1.amazonaws.com/'
probe = requests.get(archive, params={'list-type': 2, 'prefix': 'dr1/level2/', 'max-keys': 1}, timeout=(8, 20))
assert probe.status_code == 200 and b'ListBucketResult' in probe.content, 'Public NASA S3 is unreachable from this Kaggle session.'
print('Public NASA S3 listing reachable. CPU affinity:', len(os.sched_getaffinity(0)),
      'Scratch free MiB:', shutil.disk_usage('/tmp').free // 1024**2)
meminfo = Path('/proc/meminfo').read_text().splitlines()[0]
print(meminfo, 'Worker memory limit MiB:', RAM_MIB)

inputs = Path('/kaggle/input')
sources = list(inputs.rglob('slowtrace-kaggle-source.zip'))
expanded = [p.parent.parent for p in inputs.rglob('science_worker.py') if p.parent.name == 'cloud-survey']
assert len(sources) + len(expanded) == 1, 'Attach exactly one copy of the SLOWTRACE X source dataset.'
with tempfile.TemporaryDirectory(prefix='slowtrace-code-', dir='/tmp') as directory:
    root = Path(directory)
    if sources:
        with ZipFile(sources[0]) as source:
            source.extractall(root)
    else:
        shutil.copytree(expanded[0], root, dirs_exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--disable-pip-version-check',
                    '-r', str(root / 'requirements.txt')], check=True, timeout=300)
    secrets = UserSecretsClient()
    child_env = os.environ.copy()
    for name in ('SLOWTRACE_SUPABASE_URL', 'SLOWTRACE_SUPABASE_SECRET_KEY'):
        child_env[name] = secrets.get_secret(name)
    child_env['SLOWTRACE_PILOT_VERIFIED'] = 'yes' if PILOT_VERIFIED else 'no'
    command = [sys.executable, str(root / 'cloud-survey' / 'worker.py'), '--mode', MODE,
               '--batch', str(BATCH_PRODUCTS), '--batches', str(BATCHES), '--minutes', str(MINUTES),
               '--ram-mib', str(RAM_MIB), '--scratch-mib', str(SCRATCH_MIB), '--transfer-mib', str(TRANSFER_MIB)]
    if MODE == 'pilot' and PILOT_STOP_AFTER is not None:
        command += ['--pilot-stop-after', str(PILOT_STOP_AFTER)]
    subprocess.run(command, check=True, env=child_env, cwd=root, timeout=min(8 * 3600, MINUTES * 60 + 60))